In [ ]:
!pip install -q gradio spaces

In [ ]:
from huggingface_hub import login

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("hf_auth_token")

login(secret_value_0)
print(secret_value_0)

In [ ]:
IMAGE_SIZE=768
NUM_STEPS=28

prompt = "A dragon standing on a rock."
concepts = ['dragon', 'rock', 'sky']

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
def clear_mem():
    import torch
    import gc

    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
import torch
from diffusers import StableDiffusion3Pipeline

pipe = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3-medium-diffusers",
    text_encoder_3=None,
    tokenizer_3=None,
    torch_dtype=torch.float16
).to(device)


In [ ]:
# for name, comp in pipe.components.items():
#     if name=='transformer':
#         print(f"Name: {name}\nComponent: {comp}")

In [ ]:
# import inspect

# print(inspect.getsource(StableDiffusion3Pipeline.encode_prompt))

In [ ]:
from typing import Optional
import torch 

def encode_concepts(
        pipeline,
        concepts: list[str] = None,
        num_images_per_prompt: int = 1,
        clip_skip: Optional[int] = None,
        max_sequence_length: int = 256,
        device: Optional[torch.device] = None,
    ):
        device = device or self._execution_device

        # Join the concepts together into a prompt
        prompt = " ".join(concepts)

        prompt = [prompt] if isinstance(prompt, str) else prompt

        prompt_embed, _ = pipeline._get_clip_prompt_embeds(
            prompt=prompt,
            device=device,
            num_images_per_prompt=num_images_per_prompt,
            clip_skip=clip_skip,
            clip_model_index=0,
        )
        prompt_2_embed, _ = pipeline._get_clip_prompt_embeds(
            prompt=prompt,
            device=device,
            num_images_per_prompt=num_images_per_prompt,
            clip_skip=clip_skip,
            clip_model_index=1,
        )
        clip_prompt_embeds = torch.cat([prompt_embed, prompt_2_embed], dim=-1)

        t5_prompt_embed = pipeline._get_t5_prompt_embeds(
            prompt=prompt,
            num_images_per_prompt=num_images_per_prompt,
            max_sequence_length=max_sequence_length,
            device=device,
        )

        clip_prompt_embeds = torch.nn.functional.pad(
            clip_prompt_embeds,
            (0, t5_prompt_embed.shape[-1] - clip_prompt_embeds.shape[-1]),
        )

        prompt_embeds = torch.cat([clip_prompt_embeds, t5_prompt_embed], dim=-2)
        # pooled_prompt_embeds = torch.cat([pooled_prompt_embed, pooled_prompt_2_embed], dim=-1)
        prompt_embeds = prompt_embeds[
            :, : len(concepts) + 1
        ]  # Leave room for a padding token

        return prompt_embeds

In [ ]:
concept_embeddings = encode_concepts(
    pipeline=pipe, 
    concepts=concepts,
    device=pipe.device,
    num_images_per_prompt=1
)

prompt_embeds, negative_prompt_embeds, pooled_prompt_embeds, negative_pooled_prompt_embeds = pipe.encode_prompt(
    prompt=prompt,
    prompt_2=prompt,
    prompt_3=prompt,
    device=pipe.device,
    num_images_per_prompt=1,
    do_classifier_free_guidance=True,
)

# print("Prompt out shape:", prompt_output.shape)
print("prompt embeds:",prompt_embeds.shape)
print("pooled prompt embeds:",pooled_prompt_embeds.shape)
print("Concept Embeddings Shape:", concept_embeddings.shape)
concept_embeds = pipe.transformer.context_embedder(concept_embeddings)
print(concept_embeds.shape)

In [ ]:
# del pipe.text_encoder
# del pipe.text_encoder_2

# clear_mem()

In [ ]:
# print(inspect.getsource(type(pipe.transformer.transformer_blocks[13].attn.processor)))

In [ ]:
from diffusers.models.attention_processor import Attention
import torch.nn.functional as F

class CustomAttnProcessor:
    """Attention processor used typically in processing the SD3-like self-attention projections."""

    def __init__(self):
        self.store_out = {
            'concept_out': [],
            'image_out': [],
            # 'text_out': []
        }
        if not hasattr(F, "scaled_dot_product_attention"):
            raise ImportError("JointAttnProcessor2_0 requires PyTorch 2.0, to use it, please upgrade PyTorch to 2.0.")

    def __call__(
        self,
        attn: Attention,
        hidden_states: torch.FloatTensor,
        encoder_hidden_states: torch.FloatTensor = None,
        attention_mask: torch.FloatTensor | None = None,
        concept_hidden_states: torch.FloatTensor | None = None,
        *args,
        **kwargs,
    ) -> torch.FloatTensor:
        # print(concept_hidden_states is not None)
        residual = hidden_states

        batch_size = hidden_states.shape[0]

        # `sample` projections.
        query = attn.to_q(hidden_states)
        key = attn.to_k(hidden_states)
        value = attn.to_v(hidden_states)

        inner_dim = key.shape[-1]
        head_dim = inner_dim // attn.heads

        img_q = query.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)
        img_k = key.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)
        img_v = value.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)

        if attn.norm_q is not None:
            img_q = attn.norm_q(img_q)
        if attn.norm_k is not None:
            img_k = attn.norm_k(img_k)

        # `context` projections.
        if encoder_hidden_states is not None:
            text_q = attn.add_q_proj(encoder_hidden_states)
            text_k = attn.add_k_proj(encoder_hidden_states)
            text_v = attn.add_v_proj(encoder_hidden_states)

            text_q = text_q.view(
                batch_size, -1, attn.heads, head_dim
            ).transpose(1, 2)
            text_k = text_k.view(
                batch_size, -1, attn.heads, head_dim
            ).transpose(1, 2)
            text_v = text_v.view(
                batch_size, -1, attn.heads, head_dim
            ).transpose(1, 2)

            if attn.norm_added_q is not None:
                text_q = attn.norm_added_q(text_q)
            if attn.norm_added_k is not None:
                text_k = attn.norm_added_k(text_k)

        out_c = None
        if concept_hidden_states is not None:
            concept_q = attn.add_q_proj(concept_hidden_states)
            concept_k = attn.add_k_proj(concept_hidden_states)
            concept_v = attn.add_v_proj(concept_hidden_states)

            concept_q = concept_q.view(
                batch_size, -1, attn.heads, head_dim
            ).transpose(1, 2)
            concept_k = concept_k.view(
                batch_size, -1, attn.heads, head_dim
            ).transpose(1, 2)
            concept_v = concept_v.view(
                batch_size, -1, attn.heads, head_dim
            ).transpose(1, 2)

            if attn.norm_added_q is not None:
                concept_q = attn.norm_added_q(concept_q)
            if attn.norm_added_k is not None:
                concept_k = attn.norm_added_k(concept_k)


            k_xc = torch.cat([img_k, concept_k], dim=2)
            v_xc = torch.cat([img_v, concept_v], dim=2)

            out_c = F.scaled_dot_product_attention(concept_q, k_xc, v_xc, dropout_p=0.0, is_causal=False)
            out_c = out_c.transpose(1, 2).reshape(batch_size, -1, attn.heads * head_dim)
            out_c = out_c.to(concept_q.dtype)

            # self.store_out['concept_out'].append(out_c.detach().cpu())
            self.store_out['concept_out'].append(out_c[1].detach().cpu().half())
            

        if encoder_hidden_states is not None:
            query = torch.cat([img_q, text_q], dim=2)
            key = torch.cat([img_k, text_k], dim=2)
            value = torch.cat([img_v, text_v], dim=2)

            # print("query shape after cat",query.shape)
            # print("key shape after cat",key.shape)
        
        hidden_states = F.scaled_dot_product_attention(query, key, value, dropout_p=0.0, is_causal=False)
        hidden_states = hidden_states.transpose(1, 2).reshape(batch_size, -1, attn.heads * head_dim)
        hidden_states = hidden_states.to(query.dtype)

        if encoder_hidden_states is not None:
            # Split the attention outputs.
            hidden_states, encoder_hidden_states = (
                hidden_states[:, : residual.shape[1]],
                hidden_states[:, residual.shape[1] :],
            )
            # self.store_out['image_out'].append(hidden_states.detach().cpu())
            self.store_out['image_out'].append(hidden_states[1].detach().cpu().half())

            # self.store_out['text_out'].append(encoder_hidden_states.detach().cpu())
            if not attn.context_pre_only:
                encoder_hidden_states = attn.to_add_out(encoder_hidden_states)

        # linear proj
        hidden_states = attn.to_out[0](hidden_states)
        # dropout
        hidden_states = attn.to_out[1](hidden_states)

        if concept_hidden_states is not None:
            return hidden_states, encoder_hidden_states, out_c
        elif encoder_hidden_states is not None:
            return hidden_states, encoder_hidden_states
        else:
            return hidden_states

In [ ]:
# print(inspect.getsource(type(pipe.transformer.transformer_blocks[10])))

In [ ]:
from typing import Any

def transformer_block_forward_patch(
    self,
    hidden_states: torch.FloatTensor,
    encoder_hidden_states: torch.FloatTensor,
    temb: torch.FloatTensor,
    joint_attention_kwargs: dict[str, Any] | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    joint_attention_kwargs = joint_attention_kwargs or {}
    concept_hidden_states = joint_attention_kwargs.get("concept_hidden_states", None)
    
    if self.use_dual_attention:
        norm_hidden_states, gate_msa, shift_mlp, scale_mlp, gate_mlp, norm_hidden_states2, gate_msa2 = self.norm1(
            hidden_states, emb=temb
        )
    else:
        norm_hidden_states, gate_msa, shift_mlp, scale_mlp, gate_mlp = self.norm1(hidden_states, emb=temb)

    if self.context_pre_only:
        norm_encoder_hidden_states = self.norm1_context(encoder_hidden_states, temb)
    else:
        norm_encoder_hidden_states, c_gate_msa, c_shift_mlp, c_scale_mlp, c_gate_mlp = self.norm1_context(
            encoder_hidden_states, emb=temb
        )

    if concept_hidden_states is not None:
        norm_concept_hidden_states, _, _, _, _ = self.norm1_context(
            concept_hidden_states, emb=temb
        )
        joint_attention_kwargs["concept_hidden_states"] = norm_concept_hidden_states
    
    # Attention.
    if concept_hidden_states is not None:
        attn_output, context_attn_output, concept_attn_output = self.attn(
            hidden_states=norm_hidden_states,
            encoder_hidden_states=norm_encoder_hidden_states,
            **joint_attention_kwargs,
        )
    else:
        attn_output, context_attn_output = self.attn(
            hidden_states=norm_hidden_states,
            encoder_hidden_states=norm_encoder_hidden_states,
            **joint_attention_kwargs,
        )

    # Process attention outputs for the `hidden_states`.
    attn_output = gate_msa.unsqueeze(1) * attn_output
    hidden_states = hidden_states + attn_output

    if self.use_dual_attention:
        attn_output2 = self.attn2(hidden_states=norm_hidden_states2, **joint_attention_kwargs)
        attn_output2 = gate_msa2.unsqueeze(1) * attn_output2
        hidden_states = hidden_states + attn_output2

    norm_hidden_states = self.norm2(hidden_states)
    norm_hidden_states = norm_hidden_states * (1 + scale_mlp[:, None]) + shift_mlp[:, None]
    if self._chunk_size is not None:
        # "feed_forward_chunk_size" can be used to save memory
        ff_output = _chunked_feed_forward(self.ff, norm_hidden_states, self._chunk_dim, self._chunk_size)
    else:
        ff_output = self.ff(norm_hidden_states)
    ff_output = gate_mlp.unsqueeze(1) * ff_output

    hidden_states = hidden_states + ff_output

    # Process attention outputs for the `encoder_hidden_states`.
    if self.context_pre_only:
        encoder_hidden_states = None
    else:
        context_attn_output = c_gate_msa.unsqueeze(1) * context_attn_output
        encoder_hidden_states = encoder_hidden_states + context_attn_output

        norm_encoder_hidden_states = self.norm2_context(encoder_hidden_states)
        norm_encoder_hidden_states = norm_encoder_hidden_states * (1 + c_scale_mlp[:, None]) + c_shift_mlp[:, None]
        if self._chunk_size is not None:
            # "feed_forward_chunk_size" can be used to save memory
            context_ff_output = _chunked_feed_forward(
                self.ff_context, norm_encoder_hidden_states, self._chunk_dim, self._chunk_size
            )
        else:
            context_ff_output = self.ff_context(norm_encoder_hidden_states)
        encoder_hidden_states = encoder_hidden_states + c_gate_mlp.unsqueeze(1) * context_ff_output
        
        if concept_hidden_states is not None and not self.context_pre_only:
            concept_attn_output = c_gate_msa.unsqueeze(1) * concept_attn_output
            concept_hidden_states = concept_hidden_states + concept_attn_output
    
            norm_concept_hidden_states = self.norm2_context(concept_hidden_states)
            
            norm_concept_hidden_states = norm_concept_hidden_states * (1 + c_scale_mlp[:, None]) + c_shift_mlp[:, None]
            
            concept_ff_output = self.ff_context(norm_concept_hidden_states)
            concept_hidden_states = concept_hidden_states + c_gate_mlp.unsqueeze(1) * concept_ff_output
            
            joint_attention_kwargs["concept_hidden_states"] = concept_hidden_states
    
    return encoder_hidden_states, hidden_states


In [ ]:
import types

for i in range(23):
    block = pipe.transformer.transformer_blocks[i]
    
    block.attn.processor = custom_processor
    block.forward = types.MethodType(transformer_block_forward_patch, block)

In [ ]:
# concepts = ['cat', 'board']
# concept_embeddings = encode_concepts(
#     pipeline=pipe, 
#     concepts=concepts,
#     device=pipe.device
# )
# print("Concept Embeddings Shape:", concept_embeddings.shape)

In [ ]:
# test = {'t': []}
# test['t'].append(1)
# test['t'].append(2)
# print(test)

In [ ]:

output = pipe(
    prompt_embeds=prompt_embeds,
    negative_prompt_embeds=negative_prompt_embeds,
    pooled_prompt_embeds=pooled_prompt_embeds,
    negative_pooled_prompt_embeds=negative_pooled_prompt_embeds,
    num_inference_steps=NUM_STEPS,
    guidance_scale=7.0,
    height=IMAGE_SIZE,
    width=IMAGE_SIZE,
    joint_attention_kwargs={"concept_hidden_states": concept_embeds}
)

concept_out_list = custom_processor.store_out['concept_out']
image_out_list = custom_processor.store_out['image_out']

# custom_processor.store_out['concept_out'].clear()
# custom_processor.store_out['image_out'].clear()
# clear_mem()

print("concept out: ",len(concept_out_list))
print("image out: ",len(image_out_list))

for ox, oc in zip(image_out_list, concept_out_list):
    print(ox.shape)
    print(oc.shape)
    break

del concept_out_list
del image_out_list

In [ ]:
# del prompt_embeds, negative_prompt_embeds, pooled_prompt_embeds, negative_pooled_prompt_embeds,  concept_embeds

In [ ]:
img_info = {
    'vae_scale_factor': pipe.vae_scale_factor,
    'patch_size': pipe.patch_size
}
print(img_info)

In [ ]:
# del pipe

# clear_mem()

In [ ]:
concept_vector = torch.stack(custom_processor.store_out['concept_out']).view(NUM_STEPS, 23, 1, -1, 1536)
image_vector = torch.stack(custom_processor.store_out['image_out']).view(NUM_STEPS, 23, 1, -1, 1536)


print(concept_vector.shape)
print(image_vector.shape)

In [ ]:
print(f"List length: {len(custom_processor.store_out['concept_out'])}")

In [ ]:
custom_processor.store_out['concept_out'].clear()
custom_processor.store_out['image_out'].clear()

In [ ]:
# del custom_processor
# clear_mem()

In [ ]:
from typing import List
def compute_heatmaps(
        img_info,
        concept_vectors: torch.Tensor,
        image_vectors: torch.Tensor,
        height: int,
        width: int,
        layer_indices: Optional[List[int]] = None,
        softmax: bool = True,
        average_over_timesteps: bool = False,
    ) -> torch.Tensor:
        """
        Compute heatmaps from concept and image vectors.

        Args:
            concept_vectors: Shape (layers, batch, concepts, dims) or (timesteps, layers, batch, concepts, dims)
            image_vectors: Shape (layers, batch, pixels, dims) or (timesteps, layers, batch, pixels, dims)
            height: Image height in pixels
            width: Image width in pixels
            layer_indices: Which layers to use (default: all)
            softmax: Whether to apply softmax normalization
            average_over_timesteps: Whether vectors have timestep dimension

        Returns:
            Tensor of shape (num_concepts, h, w)
        """
        print("initial vec:")
        print("concept", concept_vectors.shape)
        print("img", image_vectors.shape)
        # Handle timestep dimension if present
        if average_over_timesteps and len(concept_vectors.shape) == 5:
            # Average across timesteps first
            concept_vectors = torch.mean(concept_vectors, dim=0)
            image_vectors = torch.mean(image_vectors, dim=0)

        print("after average over time:")
        print("concept:",concept_vectors.shape)
        print("image:",image_vectors.shape)

        # Shape is now (layers, batch, tokens, dims)
        num_layers = concept_vectors.shape[0]

        # Default to all layers
        if layer_indices is None:
            layer_indices = list(range(num_layers))

        # Drop the padding token (first token)
        concept_vectors = concept_vectors[:, :, 1:]
        print("dropping padding token in concept:", concept_vectors.shape)

        # Normalize concept vectors
        concept_vectors = concept_vectors / (concept_vectors.norm(dim=-1, keepdim=True) + 1e-8)

        # Compute heatmaps via einsum
        # concept_vectors: (layers, batch, concepts, dims)
        # image_vectors: (layers, batch, pixels, dims)
        heatmaps = torch.einsum(
            "lbcd,lbpd->lbcp",
            concept_vectors.float(),
            image_vectors.float()
        )
        # heatmaps: (layers, batch, concepts, pixels)
        print("after dot product",heatmaps.shape)
        # Apply softmax across concepts
        if softmax:
            heatmaps = torch.nn.functional.softmax(heatmaps, dim=-2)

        # Select specified layers
        heatmaps = heatmaps[layer_indices]

        # Average across layers
        heatmaps = heatmaps.mean(dim=0)  # (batch, concepts, pixels)
        print("avg across layers",heatmaps.shape)

        # Remove batch dimension (assuming batch=1)
        heatmaps = heatmaps[0]  # (concepts, pixels)

        # Reshape to spatial dimensions
        # For SD3 with 1024x1024 image: 64x64 tokens (patch_size=2, vae_scale_factor=8)
        h_tokens = height // (img_info['vae_scale_factor'] * img_info['patch_size'])
        w_tokens = width // (img_info['vae_scale_factor'] * img_info['patch_size'])

        print("height:",h_tokens)
        print("width:",w_tokens)

        heatmaps = heatmaps.view(-1, h_tokens, w_tokens)

        return heatmaps

In [ ]:
heatmaps = compute_heatmaps(
    concept_vectors=concept_vector,
    image_vectors=image_vector,
    height=IMAGE_SIZE,
    width=IMAGE_SIZE,
    # layer_indices=[i for i in range(10,20)],
    average_over_timesteps=True
)

In [ ]:
# del concept_vector
# del image_vector

# clear_mem()

In [ ]:
heatmaps.shape
output.images[0].save("output.png")

In [ ]:
import matplotlib.pyplot as plt 

cols = (len(concepts) + 2) // 2
fig, axes = plt.subplots(2, cols, figsize=(12, 5))

img = plt.imread("output.png")

for idx in range(len(concepts) + 1):
    row = idx // cols
    col = idx % cols

    if idx == 0:
        axes[row][col].imshow(img)
        axes[row][col].set_title("Original Image")
    else:
        hm = heatmaps[idx - 1].detach().cpu().numpy()
        hm = (hm - hm.min()) / (hm.max() - hm.min() + 1e-8)

        axes[row][col].imshow(hm, cmap="plasma")
        axes[row][col].set_title(f"Concept: {concepts[idx - 1]}")

    axes[row][col].axis("off")

for idx in range(len(concepts) + 1, 2 * cols):
    row = idx // cols
    col = idx % cols
    axes[row][col].axis("off")

plt.tight_layout()
plt.savefig(f'output_plot{'_'.join(prompt.split(' '))}.png')
plt.show()

In [1]:
from huggingface_hub import login

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("hf_auth_token")

login(secret_value_0)
print(bool(secret_value_0))

from diffusers import StableDiffusion3Pipeline
import torch
pipe = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3-medium-diffusers",
    text_encoder_3=None,
    tokenizer_3=None,
    torch_dtype=torch.float16,
)

True


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]